# American Community Survey data

<style>
blockquote:has(.notebook-admonition-title) {
  --notebook-admonition-color: var(--color-admonition-title--note, #087fc7);
  --notebook-admonition-title-background:
    var(--color-admonition-title-background--note, rgba(8, 127, 199, 0.18));
  background: var(--color-admonition-background, transparent);
  border: 0;
  border-left: 0.2rem solid var(--notebook-admonition-color);
  border-radius: 0.2rem;
  box-shadow: 0 0.2rem 0.5rem rgba(0, 0, 0, 0.05), 0 0 0.0625rem rgba(0, 0, 0, 0.1);
  font-size: var(--admonition-font-size, 0.8125rem);
  margin: 1rem auto;
  overflow: hidden;
  padding: 0 0.5rem 0.5rem;
}
blockquote p:has(> .notebook-admonition-title) {
  background: var(--notebook-admonition-title-background);
  font-size: var(--admonition-title-font-size, 0.8125rem);
  font-weight: 500;
  line-height: 1.3;
  margin: 0 -0.5rem 0.5rem;
  padding: 0.4rem 0.5rem 0.4rem 2rem;
  position: relative;
}
blockquote p:has(> .notebook-admonition-title)::before {
  color: var(--notebook-admonition-color);
  content: "✎";
  left: 0.65rem;
  position: absolute;
}
.notebook-admonition-title {
  font-weight: inherit;
}
table:not(.dataframe) {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
  border-collapse: collapse;
}
table:not(.dataframe) th,
table:not(.dataframe) td {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
}
</style>

<div style="text-align: center;"><a class="sd-sphinx-override sd-btn sd-text-wrap sd-btn-primary reference external" href="https://www.dropbox.com/scl/fo/s22x9phl0hldiakn8nbuz/ABKfxHBaak5ra3eBGkNFWMM?rlkey=igpo7qi07oz5tfgjki317o79t&amp;st=gcxkicnc&amp;dl=1">Download tutorial data</a></div>

The decennial census ([decennial guide](decennial.ipynb)) counts everyone every ten years. The
Census Bureau publishes the American Community Survey (ACS) annually from a rolling sample,
with estimates accompanied by margins of error. The ACS is also the main source of citizen 
voting-age population (CVAP).

As in the other data guides, live calls appear as copyable blocks and small committed extracts
(Georgia counties, 2023 5-year survey) stand in for the responses.

In [ ]:
from pathlib import Path

import pandas as pd

from gerrytools.data import ACSVAPTableInfo

census_dir = Path("data/census")
pd.set_option("display.max_columns", None)

## The everyday call

`acs(state, geometry, year)` defaults to three tables, total population, voting-age population
(VAP), and citizen voting-age population (CVAP), and returns a pair of frames: estimates and
margins of error. The default 5-year product (`survey="acs5"`) is available for small
geographies that the 1-year product does not cover.

```python
import us

from gerrytools.data import acs

estimates, margins = acs(us.states.GA, "county", 2023)
```

In [ ]:
estimates = pd.read_csv(
    census_dir / "ga_county_acs5_2023_estimates.csv", dtype={"GEOID": "string"}, index_col="GEOID"
)
margins = pd.read_csv(
    census_dir / "ga_county_acs5_2023_moe.csv", dtype={"GEOID": "string"}, index_col="GEOID"
)
print("estimates:", estimates.shape, "| margins:", margins.shape)
estimates[
    ["total_pop_acs5_23", "total_vap_acs5_23", "black_vap_acs5_23", "black_cvap_acs5_23"]
].head()

Each column name records the demographic group, the measure (`pop`/`vap`/`cvap`), the survey
(`acs5`), and the two-digit year. The frame is GEOID-indexed, so it joins straight onto a
shapefile or a decennial frame without colliding with the `_20` decennial columns.

## Margins of error

Every estimate has a companion margin in the second frame: the estimate column
`black_vap_acs5_23` pairs with `black_vap_moe_acs5_23` (the name gains a `_moe`).

In [ ]:
with_margin = pd.DataFrame(
    {
        "black_vap": estimates["black_vap_acs5_23"],
        "black_vap_margin": margins["black_vap_moe_acs5_23"],
    }
)
with_margin.head()

## Choosing tables

Pass `tables=[...]` to fetch exactly what you want. The available table definitions:

| Class | Contents | Census base table |
| --- | --- | --- |
| `ACSTotPopTableInfo` | total population | B01001 |
| `ACSRacePopTableInfo` | population by race alone | B02001 |
| `ACSAgeTableInfo` | sex-by-age cells and combined age bands | B01001 |
| `ACSVAPTableInfo` | voting-age population by race | B05003 and its race iterations |
| `ACSCVAPTableInfo` | citizen voting-age population by race | B05003 and its race iterations |
| `ACSHispByRaceTableInfo` | Hispanic origin by race | B03002 |

```python
from gerrytools.data import ACSHispByRaceTableInfo, ACSVAPTableInfo

table_est, table_moe = acs(
    us.states.GA, "county", 2023, tables=[ACSVAPTableInfo(), ACSHispByRaceTableInfo()]
)
```

In [ ]:
table_est = pd.read_csv(
    census_dir / "ga_county_acs5_2023_vap_hisp_estimates.csv",
    dtype={"GEOID": "string"},
    index_col="GEOID",
)
print("columns from both tables:", table_est.shape[1])
table_est.head()

## The CVAP shortcut

`cvap()` is a shorter spelling of `acs(..., tables=[ACSCVAPTableInfo()])` for analyses that
need only the ACS CVAP table.

```python
from gerrytools.data import cvap

cvap_est, cvap_moe = cvap(us.states.GA, "county", 2023)
```

In [ ]:
cvap_est = pd.read_csv(
    census_dir / "ga_county_cvap_2023_estimates.csv", dtype={"GEOID": "string"}, index_col="GEOID"
)
cvap_est[
    ["total_cvap_acs5_23", "white_cvap_acs5_23", "black_cvap_acs5_23", "hispanic_cvap_acs5_23"]
].head()

## Raw, ungrouped variables with `acs_full()`

`acs()` condenses each table: it sums the underlying age/sex/citizenship cells into one number
per group. When you need those cells themselves, for a custom aggregation or to see the
male/female split, use `acs_full()`. It takes a single table and returns the ungrouped variables
under long descriptive names.

```python
from gerrytools.data import acs_full

full_est, full_moe = acs_full(us.states.GA, "county", 2023, ACSVAPTableInfo())
```

In [ ]:
full_est = pd.read_csv(
    census_dir / "ga_county_acs5_2023_vap_full_estimates.csv",
    dtype={"GEOID": "string"},
    index_col="GEOID",
)
print("acs_full keeps every cell:", full_est.shape[1], "columns")
full_est[["total_vap_est_male_acs5_23", "total_vap_est_female_acs5_23"]].head()

Pass `rename_columns=False` to keep the raw Census variable codes instead, which is handy when you
want to map them yourself. The table object knows the mapping either way:

In [ ]:
vap_table = ACSVAPTableInfo()
long_names = vap_table.construct_long_names(year=2023)
list(long_names.items())[:4]

### Tables the shorteners do not know

`ACSTableInfo` can describe any ACS detailed table, not just the built-ins: give it the base
table code and the variable indices you want. Each info object describes exactly one base
table; to pull from several, build one object per base table and pass them together in
`tables=`. The built-in tables carry an `index_to_name_dict`
that turns each index into a descriptive name; a custom table usually has no such entries. When
an index has no name and the table has more than one variable, the long name falls back to the
zero-padded variable index. The columns therefore remain unique and traceable to the raw
Census variables rather than collapsing onto one repeated name.

```python
from frozendict import frozendict

from gerrytools.data import ACSTableInfo

commute = ACSTableInfo(
    table_name="commute",
    base_table_strings=("B08301",),  # means of transportation to work
    table_indices=(1, 2, 3, 4),
    groups_tup=("",),
    index_to_name_dict=frozendict(),
)
commute_est, commute_moe = acs_full(us.states.GA, "state", 2023, commute)
```

In [ ]:
from frozendict import frozendict

from gerrytools.data import ACSTableInfo

commute = ACSTableInfo(
    table_name="commute",
    base_table_strings=("B08301",),
    table_indices=(1, 2, 3, 4),
    groups_tup=("",),
    index_to_name_dict=frozendict(),
)
print(
    "fallback long names:",
    list(commute.construct_long_names(year=2023, source_suffix="acs5").values()),
)

commute_est = pd.read_csv(
    census_dir / "ga_state_acs5_2023_commute_full_estimates.csv",
    dtype={"GEOID": "string"},
    index_col="GEOID",
)
commute_est

The trailing `_001` ... `_004` are the raw variable indices (`B08301_001E` and so on). Supplying
your own `index_to_name_dict` entries replaces them with descriptive names, exactly as the
built-in tables do.

## Survey period: 5-year vs 1-year

`survey` accepts `"acs5"` / `5` (the default) or `"acs1"` / `1`. The 1-year survey is fresher but
only covers geographies with at least about 65,000 people:

- `acs1` is not available below the county level (a tract or block-group `acs1` request is 
  rejected), and
- an `acs1` county query silently omits small counties on the Census side; `acs()` warns when the
  returned counties do not cover the state.

```python
est5, _ = acs(us.states.GA, "state", 2023, survey="acs5")
est1, _ = acs(us.states.GA, "state", 2023, survey="acs1")
```

In [ ]:
est5 = pd.read_csv(
    census_dir / "ga_state_acs5_2023_estimates.csv", dtype={"GEOID": "string"}, index_col="GEOID"
)
est1 = pd.read_csv(
    census_dir / "ga_state_acs1_2023_estimates.csv", dtype={"GEOID": "string"}, index_col="GEOID"
)
print("5-year total population estimate:", int(est5["total_pop_acs5_23"].iloc[0]))
print("1-year total population estimate:", int(est1["total_pop_acs1_23"].iloc[0]))

## Related

- [Block CVAP guide](block_cvap.ipynb) pushes CVAP below the tract level using decennial
  block VAP.
- [Decennial guide](decennial.ipynb) for the official counts the ACS estimates sit
  alongside.
- [Overview](overview.ipynb) for GEOID handling, merging, and key setup.
- [Data API](../../api/data.rst)